# ATP Feature Engineering — Gold Layer

Reads the Stage 1 clean parquet, engineers predictive features using PySpark **Window Functions** (the distributed-computing primitive this notebook is built around), computes a **chronological Elo rating** in pandas (inherently sequential), and emits a match-level **Gold table** ready for XGBoost.

**Why window functions matter for tennis prediction.** Tennis is a time-series problem disguised as tabular data. A player's serve percentage tonight is not independent of their serve percentage over the last ten matches — it's a *realization of a trajectory*. Window functions let us compute, for every match in parallel, the state of both players strictly *before* that match — without leaking any information from the match itself.

**Leakage discipline.** Every window used in this notebook has upper bound `-1` (strictly prior rows). The current match never contributes to its own features. This is the most important invariant in the entire pipeline — if it breaks, validation metrics become worthless.

## Section 1 — Load & reshape to player-centric format

The Stage 1 output is **match-centric**: one row per match with `winner_*` and `loser_*` columns side-by-side. This shape is awkward for window functions because "win rate of player X over last 20 matches" requires gathering rows where X appears as *either* winner or loser.

We **explode** each match into two rows — one per participant. Every match now contributes one row to each player's timeline. After this step, `Window.partitionBy('player_name')` is a natural, distributed way to compute rolling stats.

In [ ]:
from collections import defaultdict

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

INPUT_PATH  = '/Workspace/Users/f.chiesa28@ncf.edu/Octenpus/datasets/atp_matches_clean_final.parquet'
OUTPUT_PATH = '/Workspace/Users/f.chiesa28@ncf.edu/Octenpus/datasets/atp_gold.parquet'
OUTPUT_CSV  = '/Workspace/Users/f.chiesa28@ncf.edu/Octenpus/datasets/atp_gold_sample.csv'

# Databricks Serverless has no FUSE mount for /Workspace, so spark.read.parquet
# on a workspace path fails with WorkspaceLocalFileSystemInternalError.
# Load with pandas+pyarrow and hand the frame to Spark.
pdf_in = pd.read_parquet(INPUT_PATH, engine='pyarrow')
df = spark.createDataFrame(pdf_in)
df = df.withColumn('match_date', F.to_date('match_date'))
df = df.withColumn('match_id', F.concat_ws('-', F.col('tourney_id'), F.col('match_num').cast('string')))

print(f'Input matches: {df.count():,}   columns: {len(df.columns)}')

# tourney_id is needed for the split-fatigue features in Section 3d.
COMMON = ['match_id', 'match_date', 'tourney_id', 'surface', 'tourney_level',
          'round', 'best_of', 'year', 'score']

winner_side = df.select(
    *COMMON,
    F.col('winner_name').alias('player_name'),
    F.col('loser_name').alias('opponent_name'),
    F.lit(1).alias('won'),
    F.col('w_ace').alias('ace'),
    F.col('w_df').alias('df'),
    F.col('w_svpt').alias('svpt'),
    F.col('w_1stIn').alias('first_in'),
    F.col('w_1stWon').alias('first_won'),
    F.col('w_2ndWon').alias('second_won'),
    F.col('w_SvGms').alias('sv_gms'),
    F.col('w_bpSaved').alias('bp_saved'),
    F.col('w_bpFaced').alias('bp_faced'),
    F.col('w_1stServe_pct').alias('first_serve_pct'),
    F.col('w_1stWon_pct').alias('first_won_pct'),
    F.col('w_2ndWon_pct').alias('second_won_pct'),
    F.col('w_bp_conversion').alias('bp_conversion'),
    F.col('winner_hand').alias('player_hand'),
    F.col('winner_age').alias('player_age'),
    F.col('winner_rank').alias('player_rank'),
)

loser_side = df.select(
    *COMMON,
    F.col('loser_name').alias('player_name'),
    F.col('winner_name').alias('opponent_name'),
    F.lit(0).alias('won'),
    F.col('l_ace').alias('ace'),
    F.col('l_df').alias('df'),
    F.col('l_svpt').alias('svpt'),
    F.col('l_1stIn').alias('first_in'),
    F.col('l_1stWon').alias('first_won'),
    F.col('l_2ndWon').alias('second_won'),
    F.col('l_SvGms').alias('sv_gms'),
    F.col('l_bpSaved').alias('bp_saved'),
    F.col('l_bpFaced').alias('bp_faced'),
    F.col('l_1stServe_pct').alias('first_serve_pct'),
    F.col('l_1stWon_pct').alias('first_won_pct'),
    F.col('l_2ndWon_pct').alias('second_won_pct'),
    F.col('l_bp_conversion').alias('bp_conversion'),
    F.col('loser_hand').alias('player_hand'),
    F.col('loser_age').alias('player_age'),
    F.col('loser_rank').alias('player_rank'),
)

df_long = winner_side.unionByName(loser_side)

# Per-match OBSERVED serve-win percentage. This is the actual fraction of serve
# points the player won in this specific match — the ground-truth value Stage 1
# of the model will be trained against. Distinct from the *rolling* serve features
# which are leakage-free historical aggregates used as inputs.
df_long = df_long.withColumn(
    'serve_won_pct_observed',
    F.when(
        F.col('svpt').isNotNull() & (F.col('svpt') > 0),
        (F.col('first_won') + F.col('second_won')) / F.col('svpt')
    )
)

print(f'Player-match rows after explode: {df_long.count():,}   (should be ≈ 2 × input)')
print('✓ Section 1 complete — df_long has one row per (player, match), incl. observed serve %')

## Section 2 — Elo rating (chronological, pandas)

Elo is the single most predictive feature in tennis match modeling — it compresses "relative player strength as of a given date" into one number and has been shown to beat official ATP rankings for forecasting. Every serious tennis model uses some Elo variant.

**Why pandas here and not Spark.** Elo is inherently sequential: rating after match *N* depends on the rating going into match *N*, which depends on every match before it. Window functions cannot express this recurrence (they compute aggregates, not recurrences). We therefore drop into pandas for this one section.

**Leakage guard.** We record the rating for each player *before* the match's outcome is applied (`pre-match Elo`). The rating is updated *after* the row is written. This is the feature the model will see at prediction time.

Parameters: `K = 32` (standard ATP Elo K-factor), starting rating `1500`.

In [ ]:
K = 32.0
START_RATING = 1500.0

pdf = df_long.toPandas().sort_values(['match_date', 'match_id']).reset_index(drop=True)

elo = defaultdict(lambda: START_RATING)
player_elo = [None] * len(pdf)

# Group rows by match_id. Each match has exactly 2 rows (one per player).
grouped = pdf.groupby('match_id', sort=False).indices

# Walk match_ids in chronological order (pdf is already sorted, groupby preserves order).
for mid, idxs in grouped.items():
    if len(idxs) != 2:
        continue  # defensive — drop orphaned half-matches
    i1, i2 = int(idxs[0]), int(idxs[1])
    p1 = pdf.at[i1, 'player_name']
    p2 = pdf.at[i2, 'player_name']
    won1 = int(pdf.at[i1, 'won'])

    r1, r2 = elo[p1], elo[p2]
    # Pre-match rating is what the model sees
    player_elo[i1] = r1
    player_elo[i2] = r2

    exp1 = 1.0 / (1.0 + 10 ** ((r2 - r1) / 400.0))
    exp2 = 1.0 - exp1
    elo[p1] = r1 + K * (won1 - exp1)
    elo[p2] = r2 + K * ((1 - won1) - exp2)

pdf['player_elo'] = player_elo

# Back to Spark for the rest of the pipeline
df_long = spark.createDataFrame(pdf)

print(f'Unique players rated: {len(elo):,}')
print(f'Elo range observed:   {pdf["player_elo"].min():.0f}  \u2192  {pdf["player_elo"].max():.0f}')
print('\u2713 Section 2 complete \u2014 player_elo attached (pre-match rating)')

## Section 3 — Window function features

The heart of the notebook. Each sub-section computes a group of features using a `Window` partitioned by player (or player+surface, or player+opponent) and ordered by `match_date`. Spark distributes each partition to a different task — this is the distributed-computing concept the course grades us on.

**Every window has upper bound `-1`** so the current match cannot leak into its own feature.

| Sub-section | Feature group | Why it matters for prediction |
|---|---|---|
| 3a | Rolling win rates | Recent form dominates short-term outcomes; surface-specific form explains why a clay-courter loses at Wimbledon |
| 3b | Serve stats rolling | Serve is the single most stable player skill; trends reveal form spikes/injuries |
| 3c | Head-to-head | Some matchups are structural (big server vs returner); history encodes it |
| 3d | Fatigue — split | Two distinct signals: `matches_prev_tourneys_7d` (real fatigue/form carried in from last week) and `matches_this_tourney` (tournament progression / how deep this player is in the current draw). A naive "last 7 days" window conflates them |
| 3e | Days since last match | Rust vs freshness; extremes both hurt |
| 3f | Win streak | Momentum/confidence is measurable in tennis betting markets |
| 3g | Career matches played | Weighting — reliability of every other rolling feature |

In [ ]:
# ---- 3a) Rolling win rates -------------------------------------------------
w_player         = Window.partitionBy('player_name').orderBy('match_date')
w_player_surface = Window.partitionBy('player_name', 'surface').orderBy('match_date')

w_last20     = w_player.rowsBetween(-20, -1)
w_last5      = w_player.rowsBetween(-5, -1)
# 12-month surface window — range-based, in seconds since epoch
w_surface_12m = (Window.partitionBy('player_name', 'surface')
                        .orderBy(F.unix_timestamp('match_date'))
                        .rangeBetween(-365 * 86400, -1))

df_long = (df_long
    .withColumn('win_rate_last_20',     F.avg('won').over(w_last20))
    .withColumn('win_rate_last_5',      F.avg('won').over(w_last5))
    .withColumn('win_rate_surface_12m', F.avg('won').over(w_surface_12m))
)

# ---- 3b) Serve stats rolling (last 10, same surface) -----------------------
w_surface_10 = w_player_surface.rowsBetween(-10, -1)
w_player_10  = w_player.rowsBetween(-10, -1)

df_long = (df_long
    .withColumn('serve_pct_surface',   F.avg('first_serve_pct').over(w_surface_10))
    .withColumn('first_won_pct_roll',  F.avg('first_won_pct').over(w_surface_10))
    .withColumn('second_won_pct_roll', F.avg('second_won_pct').over(w_surface_10))
    .withColumn('_ace_rate_single',    F.when(F.col('svpt') > 0, F.col('ace') / F.col('svpt')))
    .withColumn('ace_rate',            F.avg('_ace_rate_single').over(w_player_10))
    .drop('_ace_rate_single')
)

print('\u2713 Section 3a + 3b complete \u2014 rolling win rates and serve stats computed')

In [ ]:
# ---- 3c) Head-to-head ------------------------------------------------------
w_h2h = (Window.partitionBy('player_name', 'opponent_name')
               .orderBy('match_date')
               .rowsBetween(Window.unboundedPreceding, -1))

df_long = (df_long
    .withColumn('h2h_wins',    F.sum('won').over(w_h2h))
    .withColumn('h2h_total',   F.count(F.lit(1)).over(w_h2h))
    .withColumn('h2h_winrate', F.when(F.col('h2h_total') > 0,
                                      F.col('h2h_wins') / F.col('h2h_total')))
)

# ---- 3d) Fatigue — split into two cleaner signals --------------------------
# Why split: a single "matches in last 7d" window conflates two distinct things:
#   (a) recent form carried in from previous tournaments (last week's final, etc.)
#   (b) progression inside the current tournament (qualifiers & prior rounds)
# (a) is a clean "fatigue/form" signal. (b) is basically "how deep is this player
# in the current draw" — useful, but a different concept. We emit both separately.
df_long = df_long.withColumn('total_sets', F.size(F.split(F.col('score'), ' ')))

w_7d_all = (Window.partitionBy('player_name')
                   .orderBy(F.unix_timestamp('match_date'))
                   .rangeBetween(-7 * 86400, -1))
w_7d_this = (Window.partitionBy('player_name', 'tourney_id')
                    .orderBy(F.unix_timestamp('match_date'))
                    .rangeBetween(-7 * 86400, -1))
w_tourney_all = (Window.partitionBy('player_name', 'tourney_id')
                        .orderBy('match_date')
                        .rowsBetween(Window.unboundedPreceding, -1))

df_long = (df_long
    .withColumn('_m7_all',  F.count(F.lit(1)).over(w_7d_all))
    .withColumn('_m7_this', F.count(F.lit(1)).over(w_7d_this))
    .withColumn('_s7_all',  F.coalesce(F.sum('total_sets').over(w_7d_all),  F.lit(0)))
    .withColumn('_s7_this', F.coalesce(F.sum('total_sets').over(w_7d_this), F.lit(0)))
    .withColumn('matches_prev_tourneys_7d', F.col('_m7_all') - F.col('_m7_this'))
    .withColumn('sets_prev_tourneys_7d',    F.col('_s7_all') - F.col('_s7_this'))
    .withColumn('matches_this_tourney',     F.count(F.lit(1)).over(w_tourney_all))
    .drop('_m7_all', '_m7_this', '_s7_all', '_s7_this')
)

print('✓ Section 3c + 3d complete — H2H and split fatigue features computed')

In [ ]:
# ---- 3e) Days since last match --------------------------------------------
df_long = (df_long
    .withColumn('last_match_date', F.lag('match_date', 1).over(w_player))
    .withColumn('days_rest', F.datediff(F.col('match_date'), F.col('last_match_date')))
    .fillna({'days_rest': 30})   # first career match → treat as well rested
    .drop('last_match_date')
)

# ---- 3f) Win streak (cumulative-sum grouping trick) -----------------------
# idea: a new streak begins after every loss (or at career start).
# streak_group is constant within a streak, so we count prior rows in the same group.
w_player_cum = w_player.rowsBetween(Window.unboundedPreceding, 0)

df_long = (df_long
    .withColumn('_won_lag', F.lag('won', 1).over(w_player))
    .withColumn('_is_fresh_streak',
                F.when(F.col('_won_lag').isNull() | (F.col('_won_lag') == 0), 1).otherwise(0))
    .withColumn('_streak_group', F.sum('_is_fresh_streak').over(w_player_cum))
)

w_streak = (Window.partitionBy('player_name', '_streak_group')
                   .orderBy('match_date')
                   .rowsBetween(Window.unboundedPreceding, -1))

df_long = (df_long
    .withColumn('win_streak', F.count(F.lit(1)).over(w_streak))
    .withColumn('win_streak', F.least(F.col('win_streak'), F.lit(20)))   # cap outliers
    .drop('_won_lag', '_is_fresh_streak', '_streak_group')
)

# ---- 3g) Career matches played -------------------------------------------
w_career = w_player.rowsBetween(Window.unboundedPreceding, -1)
df_long = df_long.withColumn('matches_played', F.count(F.lit(1)).over(w_career))

# Null-rate inspection for every new feature
NEW_FEATS = [
    'player_elo',
    'win_rate_last_20', 'win_rate_last_5', 'win_rate_surface_12m',
    'serve_pct_surface', 'first_won_pct_roll', 'second_won_pct_roll', 'ace_rate',
    'h2h_wins', 'h2h_total', 'h2h_winrate',
    'matches_prev_tourneys_7d', 'sets_prev_tourneys_7d', 'matches_this_tourney',
    'days_rest', 'win_streak', 'matches_played',
]

total = df_long.count()
null_expr = [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in NEW_FEATS]
null_row = df_long.select(*null_expr).collect()[0].asDict()
print('Null rates (feature → % null):')
for c in NEW_FEATS:
    pct = 100.0 * null_row[c] / total if total else 0.0
    print(f'  {c:<26s} {pct:6.2f}%')
print('✓ Section 3 complete — 17 window-function features attached')

## Section 4 — Reconstruct match-level Gold table

The model needs **one row per match** with features for *both* participants (`p1_*`, `p2_*`). We split `df_long` back into two copies and self-join on `match_id`.

**Who is p1?** Naive choice: `p1 = winner`. That makes the target constant-1 and useless. Instead, we assign by alphabetical order — `p1 = min(player_name, opponent_name)`. This is deterministic, independent of outcome, and produces a naturally balanced label.

In [ ]:
PLAYER_FEATS = [
    'player_elo', 'player_rank', 'player_age', 'player_hand',
    'win_rate_last_20', 'win_rate_last_5', 'win_rate_surface_12m',
    'serve_pct_surface', 'first_won_pct_roll', 'second_won_pct_roll', 'ace_rate',
    'h2h_wins', 'h2h_total', 'h2h_winrate',
    'matches_prev_tourneys_7d', 'sets_prev_tourneys_7d', 'matches_this_tourney',
    'days_rest', 'win_streak', 'matches_played',
    'serve_won_pct_observed',  # per-match observed serve% — Stage 1 training target
    'won',
]

CONTEXT_COLS = ['match_id', 'match_date', 'surface', 'tourney_level', 'round', 'best_of', 'year']

base = df_long.select(*CONTEXT_COLS, 'player_name', 'opponent_name', *PLAYER_FEATS)

# Deterministic p1 / p2 assignment
side = base.withColumn(
    'side',
    F.when(F.col('player_name') < F.col('opponent_name'), F.lit('p1')).otherwise(F.lit('p2'))
)

def prefixed(side_df, prefix):
    renamed = side_df
    for c in PLAYER_FEATS + ['player_name']:
        renamed = renamed.withColumnRenamed(c, f'{prefix}_{c.replace("player_", "")}')
    return renamed.drop('opponent_name', 'side')

p1 = prefixed(side.filter(F.col('side') == 'p1'), 'p1')
p2 = (side.filter(F.col('side') == 'p2')
          .select('match_id', *PLAYER_FEATS, 'player_name')
         )
# rename p2 columns (drop context cols from p2 — keep only p1's copy)
for c in PLAYER_FEATS + ['player_name']:
    p2 = p2.withColumnRenamed(c, f'p2_{c.replace("player_", "")}')

gold = p1.join(p2, on='match_id', how='inner')

# Target: did p1 win?
gold = gold.withColumn('winner', F.col('p1_won').cast(IntegerType())).drop('p1_won', 'p2_won')

# Difference features (symmetric pairs are the strongest predictors)
gold = (gold
    .withColumn('elo_diff',              F.col('p1_elo')                      - F.col('p2_elo'))
    .withColumn('rank_diff',             F.col('p1_rank')                     - F.col('p2_rank'))
    .withColumn('age_diff',              F.col('p1_age')                      - F.col('p2_age'))
    .withColumn('winrate_diff',          F.col('p1_win_rate_last_20')         - F.col('p2_win_rate_last_20'))
    .withColumn('serve_diff',            F.col('p1_serve_pct_surface')        - F.col('p2_serve_pct_surface'))
    .withColumn('fatigue_diff',          F.col('p1_matches_prev_tourneys_7d') - F.col('p2_matches_prev_tourneys_7d'))
    .withColumn('tourney_progress_diff', F.col('p1_matches_this_tourney')     - F.col('p2_matches_this_tourney'))
)

# Context flags
gold = (gold
    .withColumn('is_grand_slam', (F.col('tourney_level') == 'G').cast(IntegerType()))
    .withColumn('is_best_of_5',  (F.col('best_of') == 5).cast(IntegerType()))
)

n = gold.count()
print(f'Gold shape: {n:,} rows  ×  {len(gold.columns)} cols')
print('Target balance:')
gold.groupBy('winner').count().orderBy('winner').show()
print('✓ Section 4 complete — match-level Gold table reconstructed (incl. observed serve%)')

## Section 5 — Drop early rows with unreliable features

Players with fewer than 10 career matches have rolling features that are mostly null or computed over 1–3 samples (noise). Training on these rows teaches the model to rely on garbage features. We drop any match where *either* player has `matches_played < 10`.

In [ ]:
before = gold.count()
gold = gold.filter((F.col('p1_matches_played') >= 10) & (F.col('p2_matches_played') >= 10))
after = gold.count()

print(f'Dropped:   {before - after:,} rows (either player < 10 career matches)')
print(f'Remaining: {after:,} rows')
print('\u2713 Section 5 complete \u2014 thin-history matches filtered out')

## Section 6 — Feature summary & validation

Four checks before handing the table to XGBoost:

1. **Correlation with target.** A first smoke test — `elo_diff` should dominate; features that correlate at zero or unexpected sign deserve investigation.
2. **Null rates** per feature. Anything above 10% gets flagged.
3. **Descriptive stats** for key features to confirm ranges look tennis-plausible.
4. **Assertions** on the hard invariants.

In [ ]:
# 6a) Correlation ranking
NUMERIC_CANDIDATES = [f.name for f in gold.schema.fields
                      if f.dataType.simpleString() in ('double', 'int', 'bigint', 'float')
                      and f.name != 'winner']

corrs = []
for c in NUMERIC_CANDIDATES:
    try:
        v = gold.stat.corr(c, 'winner')
    except Exception:
        v = None
    if v is not None:
        corrs.append((c, v))

corrs.sort(key=lambda t: abs(t[1]) if t[1] is not None else 0, reverse=True)
print('Feature correlation with winner (|\u03c1|, descending):')
for name, v in corrs[:20]:
    print(f'  {name:<28s} {v:+.4f}')

In [ ]:
# 6b) Null rates
total = gold.count()
null_expr = [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in gold.columns]
null_row = gold.select(*null_expr).collect()[0].asDict()

print('\nNull rates per column (only cols with > 0 nulls shown):')
high_null = []
for c, n in null_row.items():
    if n == 0:
        continue
    pct = 100.0 * n / total
    marker = '  ⚠ HIGH' if pct > 10 else ''
    print(f'  {c:<32s} {pct:6.2f}%{marker}')
    if pct > 10:
        high_null.append(c)
if high_null:
    print(f'\n{len(high_null)} columns exceed the 10% null threshold — inspect before training.')

# 6c) Descriptive stats on key features (use p1_-prefixed names from Section 4)
print('\nDescriptive stats on headline features:')
gold.select('elo_diff', 'p1_win_rate_last_20', 'p1_serve_pct_surface',
            'p1_matches_prev_tourneys_7d', 'p1_matches_this_tourney',
            'p1_days_rest', 'p1_h2h_winrate').describe().show()

# 6d) Assertions
print('Assertions:')
labels_ok = set(r['winner'] for r in gold.select('winner').distinct().collect()) <= {0, 1}
print(f"  [{'PASS' if labels_ok else 'FAIL'}] winner column ∈ {{0, 1}}")

for critical in ('elo_diff', 'winner', 'surface', 'match_date'):
    ok = gold.filter(F.col(critical).isNull()).limit(1).count() == 0
    print(f"  [{'PASS' if ok else 'FAIL'}] no nulls in {critical}")

rows_ok = total > 20_000
print(f"  [{'PASS' if rows_ok else 'FAIL'}] total rows > 20,000  (actual: {total:,})")
print('✓ Section 6 complete — validation checks run')

## Section 7 — Save Gold layer

Write the full Gold table as parquet for XGBoost training, plus a 1,000-row CSV sample for manual inspection in a spreadsheet.

In [ ]:
# Databricks Serverless: distributed parquet writes to /Workspace/ hit the same
# WSFS issue as reads. Collect to pandas on the driver and write with pyarrow.
gold_pdf = gold.toPandas()
gold_pdf.to_parquet(OUTPUT_PATH, engine='pyarrow', index=False)
print(f'✓ Full Gold written to {OUTPUT_PATH}   ({len(gold_pdf):,} rows)')

# CSV sample
sample_pdf = gold_pdf.head(1000)
sample_pdf.to_csv(OUTPUT_CSV, index=False)
print(f'✓ 1,000-row CSV sample written to {OUTPUT_CSV}')

# Summary table
total_rows = len(gold_pdf)
balance = gold_pdf['winner'].value_counts().to_dict()
p0 = balance.get(0, 0); p1v = balance.get(1, 0)
pct_1 = 100.0 * p1v / total_rows if total_rows else 0.0
pct_0 = 100.0 - pct_1

surfaces = '/'.join(sorted(gold_pdf['surface'].dropna().unique().tolist()))
year_min = int(gold_pdf['year'].min())
year_max = int(gold_pdf['year'].max())

per_player_feats = len([c for c in gold_pdf.columns if c.startswith('p1_')])
diff_feats = sum(1 for c in gold_pdf.columns if c.endswith('_diff'))

summary = [
    '┌────────────────────────────────────────┐',
    '│ ATP Gold Layer — Feature Summary       │',
    '├──────────────────┬─────────────────────┤',
    f'│ Total matches    │ {total_rows:<17,} │',
    f'│ Features (p1+p2) │ {per_player_feats} per player    │',
    f'│ Diff features    │ {diff_feats:<17} │',
    f'│ Target balance   │ {pct_0:4.1f}% / {pct_1:4.1f}%     │',
    f'│ Date range       │ {year_min}–{year_max:<12} │',
    f'│ Surfaces         │ {surfaces:<17} │',
    '└──────────────────┴─────────────────────┘',
]
print('\n'.join(summary))
print('✓ Section 7 complete — Gold layer persisted, ready for XGBoost')